In [1]:
# ============================================================
# CELL 1: Install Dependencies
# ============================================================
!pip install -q ultralytics gtts pydub transformers accelerate peft bitsandbytes
!apt-get install -q ffmpeg
print("✅ Dependencies siap")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.3/41.3 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 22.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 33.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.2/98.2 kB 8.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 89.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 122.5/122.5 kB 10.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 1.29.0 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.2

In [2]:
# ============================================================
# CELL 2: Imports & Config
# ============================================================
import os, json, time, cv2, torch, gc
import numpy as np
from pathlib import Path
from datetime import datetime
from PIL import Image
from ultralytics import YOLO
from transformers import AutoProcessor, AutoModelForImageTextToText, BitsAndBytesConfig
from peft import PeftModel

# ── Konfigurasi ─────────────────────────────────────────────
VIDEO_DIR   = "/kaggle/input/datasets/allyciajoanmicheline/videos/vid" 
OUTPUT_DIR  = "/kaggle/working/pipeline_output"
YOLO_MODEL  = "/kaggle/input/models/allyciajoanmicheline/yolo-11s/coral/default/1/yolo11s.pt"  

CROP_DIR    = os.path.join(OUTPUT_DIR, "crops")
LOG_DIR     = os.path.join(OUTPUT_DIR, "logs")

for d in [OUTPUT_DIR, CROP_DIR, LOG_DIR]:
    os.makedirs(d, exist_ok=True)

# VLM yang akan diuji
VLM_CONFIGS = {
    "PaliGemma2-3B": {
        "model_name_or_path":   "google/paligemma2-3b-pt-224",
        "adapter_name_or_path": "/kaggle/input/datasets/allyciajoanmicheline/paligemma/paligemma2_3b_lora/checkpoint-1325",
    },
     "InternVL3_5-4B": {
        "model_name_or_path":   "OpenGVLab/InternVL3_5-4B-HF",
        "adapter_name_or_path": "/kaggle/input/datasets/allyciajoanmicheline/internvl35/internvl35_4b_lora/checkpoint-1325",
    },
}

PROMPT = "Tolong identifikasi kerusakan jalan di gambar ini meliputi tipe dan level bahayanya."
CONF_THRESHOLD = 0.25

print("✅ Config siap")
print(f"   Output dir: {OUTPUT_DIR}")

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
✅ Config siap
   Output dir: /kaggle/working/pipeline_output


In [3]:
# ============================================================
# CELL 3: Step 1 — YOLO Detection & Crop
# ============================================================
def deteksi_dan_crop(video_path, yolo_model, conf=0.25):
    """
    Deteksi objek dari setiap frame video menggunakan YOLO.
    Return: list of dict {frame_idx, bbox, crop_path, waktu_deteksi}
    """
    nama_video = Path(video_path).stem
    hasil_deteksi = []

    cap = cv2.VideoCapture(video_path)
    fps     = cap.get(cv2.CAP_PROP_FPS)
    total   = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    print(f"   📹 {nama_video}: {total} frames @ {fps:.1f} FPS")

    frame_idx = 0
    crop_idx  = 0

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        # Deteksi setiap N frame (skip untuk efisiensi)
        if frame_idx % 5 == 0:
            t_deteksi_mulai = time.time()
            results = yolo_model(frame, conf=conf, verbose=False)
            t_deteksi_selesai = time.time()
            waktu_deteksi = t_deteksi_selesai - t_deteksi_mulai

            for result in results:
                for box in result.boxes:
                    x1, y1, x2, y2 = map(int, box.xyxy[0].tolist())
                    conf_score = float(box.conf[0])
                    cls_id     = int(box.cls[0])
                    cls_name   = yolo_model.names[cls_id]

                    # Crop gambar
                    crop = frame[max(0,y1):y2, max(0,x1):x2]
                    if crop.size == 0:
                        continue

                    crop_path = os.path.join(
                        CROP_DIR,
                        f"{nama_video}_frame{frame_idx:05d}_box{crop_idx:03d}.jpg"
                    )
                    cv2.imwrite(crop_path, crop)

                    hasil_deteksi.append({
                        "video":          nama_video,
                        "video_path":     video_path,
                        "frame_idx":      frame_idx,
                        "timestamp_detik": frame_idx / fps if fps > 0 else 0,
                        "bbox":           [x1, y1, x2, y2],
                        "conf":           round(conf_score, 4),
                        "class_id":       cls_id,
                        "class_name":     cls_name,
                        "crop_path":      crop_path,
                        "waktu_deteksi_s": round(waktu_deteksi, 4),
                        "crop_idx":       crop_idx,
                    })
                    crop_idx += 1

        frame_idx += 1

    cap.release()
    print(f"   ✅ {len(hasil_deteksi)} deteksi dari {frame_idx} frames")
    return hasil_deteksi


# Load YOLO
print("🔫 Loading YOLO...")
yolo = YOLO(YOLO_MODEL)
print("✅ YOLO siap")

# Deteksi semua video
video_files = list(Path(VIDEO_DIR).glob("*.mp4")) + \
              list(Path(VIDEO_DIR).glob("*.avi")) + \
              list(Path(VIDEO_DIR).glob("*.mov"))

print(f"\n📁 Ditemukan {len(video_files)} video")

semua_deteksi = []
t_deteksi_total_mulai = time.time()

for vf in video_files:
    print(f"\n🎬 Memproses: {vf.name}")
    deteksi = deteksi_dan_crop(str(vf), yolo, conf=CONF_THRESHOLD)
    semua_deteksi.extend(deteksi)

t_deteksi_total_selesai = time.time()
waktu_deteksi_total = t_deteksi_total_selesai - t_deteksi_total_mulai

print(f"\n{'='*50}")
print(f"✅ Total deteksi: {len(semua_deteksi)}")
print(f"⏱️  Waktu deteksi total: {waktu_deteksi_total:.2f} detik")

🔫 Loading YOLO...
✅ YOLO siap

📁 Ditemukan 4 video

🎬 Memproses: v2.mp4
   📹 v2: 365 frames @ 30.0 FPS
   ✅ 81 deteksi dari 365 frames

🎬 Memproses: sel.mp4
   📹 sel: 67972 frames @ 60.0 FPS
   ✅ 2716 deteksi dari 67968 frames

🎬 Memproses: v1.mp4
   📹 v1: 233 frames @ 30.0 FPS
   ✅ 30 deteksi dari 233 frames

🎬 Memproses: p.mp4
   📹 p: 17047 frames @ 30.0 FPS
   ✅ 352 deteksi dari 17044 frames

✅ Total deteksi: 3179
⏱️  Waktu deteksi total: 472.04 detik


In [4]:
from huggingface_hub import login
from kaggle_secrets import UserSecretsClient
login(token=UserSecretsClient().get_secret("HF_TOKEN"))
print("✅ Login HF berhasil")

✅ Login HF berhasil


In [5]:
# ============================================================
# CELL 4: Step 2 — VLM Captioning (tiap model berurutan)
# ============================================================
from transformers import (
    AutoProcessor, AutoModelForImageTextToText, AutoTokenizer,
    BitsAndBytesConfig
)

def load_vlm(nama_model, cfg):
    """Load VLM dengan LoRA adapter dan 4-bit quantization."""
    print(f"   📦 Loading {nama_model}...")

    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16,
    )

    if "paligemma" in cfg["model_name_or_path"].lower():
        from transformers import PaliGemmaForConditionalGeneration, PaliGemmaProcessor
        processor = PaliGemmaProcessor.from_pretrained(cfg["model_name_or_path"])
        model = PaliGemmaForConditionalGeneration.from_pretrained(
            cfg["model_name_or_path"],
            quantization_config=bnb_config,
            device_map="auto"
        )
        model = PeftModel.from_pretrained(model, cfg["adapter_name_or_path"])
        model_type = "paligemma"

    elif "internvl" in cfg["model_name_or_path"].lower():
        processor = AutoProcessor.from_pretrained(
            cfg["model_name_or_path"], trust_remote_code=True
        )
        model = AutoModelForImageTextToText.from_pretrained(
            cfg["model_name_or_path"],
            quantization_config=bnb_config,
            device_map="auto",
            trust_remote_code=True
        )
        model = PeftModel.from_pretrained(model, cfg["adapter_name_or_path"])
        model_type = "internvl"

    model.eval()
    print(f"   ✅ {nama_model} loaded")
    return model, processor, model_type


def buat_caption(model, processor, model_type, crop_path, prompt):
    """Generate caption dari satu crop gambar."""
    image = Image.open(crop_path).convert("RGB")

    if model_type == "paligemma":
        inputs = processor(
            text=prompt,
            images=image,
            return_tensors="pt"
        ).to(model.device)
        with torch.no_grad():
            out = model.generate(**inputs, max_new_tokens=128)
        caption = processor.decode(out[0], skip_special_tokens=True)
        # Hapus prompt dari output
        caption = caption.replace(prompt, "").strip()

    elif model_type == "internvl":
        inputs = processor(
            text=f"<image>\n{prompt}",
            images=image,
            return_tensors="pt"
        ).to(model.device)
        with torch.no_grad():
            out = model.generate(**inputs, max_new_tokens=128)
        caption = processor.decode(out[0], skip_special_tokens=True)
        caption = caption.split(prompt)[-1].strip()

    return caption


def jalankan_captioning(nama_model, cfg, semua_deteksi):
    """Jalankan captioning untuk semua crop dengan satu model."""
    print(f"\n{'='*60}")
    print(f"🤖 Captioning dengan: {nama_model}")
    print(f"{'='*60}")

    model, processor, model_type = load_vlm(nama_model, cfg)
    hasil_caption = []

    for i, det in enumerate(semua_deteksi):
        if not os.path.exists(det["crop_path"]):
            continue

        print(f"   [{i+1}/{len(semua_deteksi)}] {Path(det['crop_path']).name}", end=" ")

        t_caption_mulai = time.time()
        try:
            caption = buat_caption(model, processor, model_type,
                                   det["crop_path"], PROMPT)
        except Exception as e:
            caption = f"[ERROR: {str(e)[:50]}]"
        t_caption_selesai = time.time()
        waktu_caption = t_caption_selesai - t_caption_mulai

        print(f"→ {waktu_caption:.2f}s")

        hasil_caption.append({
            **det,
            "model":          nama_model,
            "caption":        caption,
            "waktu_caption_s": round(waktu_caption, 4),
            "waktu_total_s":   round(det["waktu_deteksi_s"] + waktu_caption, 4),
        })

    # Bebaskan VRAM
    del model, processor
    torch.cuda.empty_cache()
    gc.collect()
    print(f"   🧹 VRAM dibersihkan")

    return hasil_caption


# ── Jalankan captioning semua model ─────────────────────────
semua_hasil = {}
t_caption_total_mulai = time.time()

for nama, cfg in VLM_CONFIGS.items():
    hasil = jalankan_captioning(nama, cfg, semua_deteksi)
    semua_hasil[nama] = hasil

t_caption_total_selesai = time.time()
waktu_caption_total = t_caption_total_selesai - t_caption_total_mulai

print(f"\n✅ Captioning selesai dalam {waktu_caption_total/60:.1f} menit")


🤖 Captioning dengan: PaliGemma2-3B
   📦 Loading PaliGemma2-3B...


preprocessor_config.json:   0%|          | 0.00/424 [00:00<?, ?B/s]

The image processor of type `SiglipImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/841 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/121 [00:00<?, ?B/s]

   ✅ InternVL3_5-4B loaded
   [1/3179] v2_frame00000_box000.jpg → 0.08s
   [2/3179] v2_frame00005_box001.jpg → 0.01s
   [3/3179] v2_frame00010_box002.jpg → 0.01s
   [4/3179] v2_frame00015_box003.jpg → 0.05s
   [5/3179] v2_frame00020_box004.jpg → 0.02s
   [6/3179] v2_frame00025_box005.jpg → 0.01s
   [7/3179] v2_frame00035_box006.jpg → 0.02s
   [8/3179] v2_frame00040_box007.jpg → 0.01s
   [9/3179] v2_frame00045_box008.jpg → 0.01s
   [10/3179] v2_frame00050_box009.jpg → 0.05s
   [11/3179] v2_frame00055_box010.jpg → 0.04s
   [12/3179] v2_frame00060_box011.jpg → 0.01s
   [13/3179] v2_frame00065_box012.jpg → 0.01s
   [14/3179] v2_frame00070_box013.jpg → 0.06s
   [15/3179] v2_frame00075_box014.jpg → 0.02s
   [16/3179] v2_frame00075_box015.jpg → 0.01s
   [17/3179] v2_frame00080_box016.jpg → 0.06s
   [18/3179] v2_frame00080_box017.jpg → 0.01s
   [19/3179] v2_frame00085_box018.jpg → 0.02s
   [20/3179] v2_frame00085_box019.jpg → 0.06s
   [21/3179] v2_frame00085_box020.jpg → 0.04s
   [22/3179] v2_

In [6]:
# ============================================================
# CELL 6: Simpan Log TXT per Model per Video
# ============================================================
def format_timestamp(detik):
    menit = int(detik // 60)
    sisa  = detik % 60
    return f"{menit:02d}:{sisa:06.3f}"


def tulis_log_txt(nama_model, hasil_list, log_dir):
    """
    Buat file TXT per model per video.
    Format: ringkasan + detail setiap bounding box.
    """
    # Kelompokkan per video
    per_video = {}
    for item in hasil_list:
        v = item["video"]
        if v not in per_video:
            per_video[v] = []
        per_video[v].append(item)

    for nama_video, items in per_video.items():
        nama_file = f"{nama_video}_{nama_model.replace(' ','_').replace('/','_')}.txt"
        jalur_log = os.path.join(log_dir, nama_file)

        with open(jalur_log, "w", encoding="utf-8") as f:
            # ── Header ──────────────────────────────────────
            f.write("="*70 + "\n")
            f.write(f"LAPORAN DETEKSI & ANALISIS KERUSAKAN JALAN\n")
            f.write("="*70 + "\n")
            f.write(f"Video       : {nama_video}\n")
            f.write(f"Model VLM   : {nama_model}\n")
            f.write(f"Tanggal     : {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
            f.write(f"Total Deteksi: {len(items)}\n")
            f.write("="*70 + "\n\n")

            # ── Ringkasan Waktu ──────────────────────────────
            if items:
                rata_deteksi = np.mean([x["waktu_deteksi_s"] for x in items])
                rata_caption = np.mean([x["waktu_caption_s"] for x in items])
                # rata_audio   = np.mean([x.get("waktu_audio_s", 0) for x in items])
                rata_total   = np.mean([x.get("waktu_total_pipeline_s", 0) for x in items])
                total_pipeline = sum([x.get("waktu_total_pipeline_s", 0) for x in items])

                f.write("RINGKASAN WAKTU PIPELINE\n")
                f.write("-"*40 + "\n")
                f.write(f"  Rata-rata waktu deteksi (YOLO)  : {rata_deteksi:.4f} detik\n")
                f.write(f"  Rata-rata waktu caption (VLM)   : {rata_caption:.4f} detik\n")
                # f.write(f"  Rata-rata waktu audio (TTS)     : {rata_audio:.4f} detik\n")
                f.write(f"  Rata-rata waktu total/deteksi   : {rata_total:.4f} detik\n")
                f.write(f"  Total waktu pipeline keseluruhan: {total_pipeline:.2f} detik\n")
                f.write("\n")

            # ── Detail per Bounding Box ──────────────────────
            f.write("DETAIL DETEKSI\n")
            f.write("-"*70 + "\n\n")

            for i, item in enumerate(items, 1):
                f.write(f"[{i:03d}] ── Bounding Box #{item['crop_idx']:03d}\n")
                f.write(f"  Timestamp Video  : {format_timestamp(item['timestamp_detik'])} "
                        f"(frame {item['frame_idx']})\n")
                f.write(f"  Kelas Deteksi    : {item['class_name']} "
                        f"(conf: {item['conf']:.4f})\n")
                f.write(f"  Koordinat BBox   : x1={item['bbox'][0]}, y1={item['bbox'][1]}, "
                        f"x2={item['bbox'][2]}, y2={item['bbox'][3]}\n")
                f.write(f"  Crop Path        : {item['crop_path']}\n")
                f.write(f"  ── Waktu:\n")
                f.write(f"     Deteksi (YOLO) : {item['waktu_deteksi_s']:.4f} detik\n")
                f.write(f"     Caption (VLM)  : {item['waktu_caption_s']:.4f} detik\n")
                # f.write(f"     Audio (TTS)    : {item.get('waktu_audio_s', 0):.4f} detik\n")
                f.write(f"     TOTAL          : {item.get('waktu_total_pipeline_s', 0):.4f} detik\n")
                f.write(f"  ── Hasil Analisis VLM:\n")
                f.write(f"     {item['caption']}\n")
                if item.get("audio_path"):
                    f.write(f"  ── Audio Output  : {item['audio_path']}\n")
                f.write("\n")

        print(f"   💾 {jalur_log}")

# Tulis log untuk semua model
print("📝 Menulis file log TXT...\n")
for nama_model, hasil_list in semua_hasil.items():
    print(f"  📁 {nama_model}:")
    tulis_log_txt(nama_model, hasil_list, LOG_DIR)

📝 Menulis file log TXT...

  📁 PaliGemma2-3B:
   💾 /kaggle/working/pipeline_output/logs/v2_PaliGemma2-3B.txt
   💾 /kaggle/working/pipeline_output/logs/sel_PaliGemma2-3B.txt
   💾 /kaggle/working/pipeline_output/logs/v1_PaliGemma2-3B.txt
   💾 /kaggle/working/pipeline_output/logs/p_PaliGemma2-3B.txt
  📁 InternVL3_5-4B:
   💾 /kaggle/working/pipeline_output/logs/v2_InternVL3_5-4B.txt
   💾 /kaggle/working/pipeline_output/logs/sel_InternVL3_5-4B.txt
   💾 /kaggle/working/pipeline_output/logs/v1_InternVL3_5-4B.txt
   💾 /kaggle/working/pipeline_output/logs/p_InternVL3_5-4B.txt


In [7]:
# ============================================================
# CELL 7: Simpan JSON & Ringkasan Akhir
# ============================================================
import pandas as pd

# Simpan JSON lengkap
for nama_model, hasil_list in semua_hasil.items():
    jalur_json = os.path.join(
        LOG_DIR,
        f"hasil_{nama_model.replace(' ','_').replace('/','_')}.json"
    )
    with open(jalur_json, "w", encoding="utf-8") as f:
        json.dump(hasil_list, f, ensure_ascii=False, indent=2)
    print(f"💾 JSON: {jalur_json}")

# ── Tabel ringkasan ──────────────────────────────────────────
print("\n" + "="*70)
print("RINGKASAN PIPELINE — SEMUA MODEL")
print("="*70)

rows = []
for nama_model, hasil_list in semua_hasil.items():
    if not hasil_list:
        continue
    rows.append({
        "Model": nama_model,
        "Total Deteksi": len(hasil_list),
        "Rata2 Deteksi (s)": round(np.mean([x["waktu_deteksi_s"] for x in hasil_list]), 4),
        "Rata2 Caption (s)": round(np.mean([x["waktu_caption_s"] for x in hasil_list]), 4),
        # "Rata2 Audio (s)":   round(np.mean([x.get("waktu_audio_s",0) for x in hasil_list]), 4),
        "Rata2 Total (s)":   round(np.mean([x.get("waktu_total_pipeline_s",0) for x in hasil_list]), 4),
        "Total Pipeline (s)":round(sum([x.get("waktu_total_pipeline_s",0) for x in hasil_list]), 2),
    })

df_summary = pd.DataFrame(rows)
print(df_summary.to_string(index=False))

df_summary.to_csv(os.path.join(OUTPUT_DIR, "ringkasan_pipeline.csv"), index=False)

# ── Waktu global ─────────────────────────────────────────────
print(f"\n⏱️  WAKTU GLOBAL")
print(f"   Deteksi YOLO (semua video) : {waktu_deteksi_total:.2f} detik")
print(f"   Captioning VLM (semua)     : {waktu_caption_total/60:.1f} menit")
# print(f"   Audio TTS (semua)          : {waktu_audio_total:.2f} detik")
print(f"\n📂 Semua output di: {OUTPUT_DIR}")
print(f"   ├── crops/        → gambar crop per deteksi")
# print(f"   ├── audio/        → file MP3 per caption")
print(f"   └── logs/         → TXT & JSON per model per video")

💾 JSON: /kaggle/working/pipeline_output/logs/hasil_PaliGemma2-3B.json
💾 JSON: /kaggle/working/pipeline_output/logs/hasil_InternVL3_5-4B.json

RINGKASAN PIPELINE — SEMUA MODEL
         Model  Total Deteksi  Rata2 Deteksi (s)  Rata2 Caption (s)  Rata2 Total (s)  Total Pipeline (s)
 PaliGemma2-3B           3179              0.015             1.5847              0.0                   0
InternVL3_5-4B           3179              0.015             0.0212              0.0                   0

⏱️  WAKTU GLOBAL
   Deteksi YOLO (semua video) : 472.04 detik
   Captioning VLM (semua)     : 86.8 menit

📂 Semua output di: /kaggle/working/pipeline_output
   ├── crops/        → gambar crop per deteksi
   └── logs/         → TXT & JSON per model per video


In [8]:
# ============================================================
# CELL 5: Step 3 — Text to Audio (gTTS)
# ============================================================
# def buat_audio(teks, output_path, lang="id"):
#    """Convert teks caption ke audio MP3."""
#    t_audio_mulai = time.time()
#    tts = gTTS(text=teks, lang=lang, slow=False)
#    tts.save(output_path)
#    waktu_audio = time.time() - t_audio_mulai
#    return round(waktu_audio, 4)


# print("🔊 Membuat audio untuk semua caption...")
# t_audio_total_mulai = time.time()

# for nama_model, hasil_list in semua_hasil.items():
#    audio_model_dir = os.path.join(AUDIO_DIR, nama_model.replace(" ", "_"))
#    os.makedirs(audio_model_dir, exist_ok=True)

#    for item in hasil_list:
#        if "[ERROR" in item["caption"]:
#            item["audio_path"]   = None
#            item["waktu_audio_s"] = 0
#            continue
#
#        nama_audio = (
#            f"{item['video']}_frame{item['frame_idx']:05d}"
#            f"_box{item['crop_idx']:03d}.mp3"
#        )
#        audio_path = os.path.join(audio_model_dir, nama_audio)
#
#        waktu_audio = buat_audio(item["caption"], audio_path)
#        item["audio_path"]    = audio_path
#        item["waktu_audio_s"] = waktu_audio
#        item["waktu_total_pipeline_s"] = round(
#            item["waktu_deteksi_s"] + item["waktu_caption_s"] + waktu_audio, 4
#        )
#
# t_audio_total_selesai = time.time()
# waktu_audio_total = t_audio_total_selesai - t_audio_total_mulai
# print(f"✅ Audio selesai dalam {waktu_audio_total:.2f} detik")